# K3-mini control-arm: 100-step Colab L4 smoke

This notebook runs the same CUDA smoke configuration as modal_app.py. Select an **L4, A100, or H100 GPU runtime** before starting; T4 and V100 runtimes are rejected.

Build the intended approximately 204 MiB (214 MB), two-shard bundle from the repo with:

    python data/prepare_colab_smoke.py --copy --zip

Then provide colab_smoke_bundle.zip (or an equivalent .tar/.tar.gz/.tgz archive). It preserves this layout:

    train_baseline.py
    config.py
    probes.py
    training_state.py
    data/manifest.jsonl
    data/val_books.json
    data/shards/manifest.json
    data/shards/gutenberg_train_*.bin
    data/shards/gutenberg_val_*.bin

Drive is recommended; the Colab uploader is also practical for this reduced bundle. Run cells in order; the training cell streams the console and preserves runs, logs, metadata, and partial failure artifacts to Drive.

In [ ]:
# 1. Reject an unsuitable runtime before downloading large packages.
import json, subprocess, sys

probe_code = """
import json, torch
if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU is visible. In Colab: Runtime > Change runtime type > GPU.')
p = torch.cuda.get_device_properties(0)
print(json.dumps({'name': p.name, 'capability': torch.cuda.get_device_capability(0),
                  'memory_bytes': p.total_memory}))
"""
GPU_INFO = json.loads(subprocess.check_output([sys.executable, "-c", probe_code], text=True))
cc = tuple(GPU_INFO["capability"])
memory_gib = GPU_INFO["memory_bytes"] / 2**30
if cc[0] < 8:
    raise RuntimeError(f"{GPU_INFO['name']} has compute capability {cc[0]}.{cc[1]}; Ampere (8.0) or newer is required.")
if GPU_INFO["memory_bytes"] < 20 * 2**30:
    raise RuntimeError(f"{GPU_INFO['name']} has only {memory_gib:.1f} GiB; at least 20 GiB is required.")
print(f"GPU guard passed: {GPU_INFO['name']}, cc {cc[0]}.{cc[1]}, {memory_gib:.1f} GiB")


In [ ]:
# 2. Select, hash, and safely extract the prepared bundle.
from pathlib import Path
import hashlib, shutil, stat, tarfile, tempfile, zipfile
from google.colab import drive, files

BUNDLE_SOURCE = "drive"  # @param ["drive", "upload"]
DRIVE_BUNDLE_PATH = "/content/drive/MyDrive/colab_smoke_bundle.zip"  # @param {type:"string"}
PERSIST_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/modded-nanogpt-smoke-runs"  # @param {type:"string"}

if BUNDLE_SOURCE not in {"drive", "upload"}:
    raise ValueError("BUNDLE_SOURCE must be 'drive' or 'upload'")
if BUNDLE_SOURCE == "drive" or PERSIST_TO_DRIVE:
    drive.mount("/content/drive", force_remount=False)

if BUNDLE_SOURCE == "drive":
    BUNDLE_PATH = Path(DRIVE_BUNDLE_PATH).expanduser()
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one prepared archive")
    upload_name, upload_bytes = next(iter(uploaded.items()))
    BUNDLE_PATH = Path("/content") / Path(upload_name).name
    BUNDLE_PATH.write_bytes(upload_bytes)
    del upload_bytes, uploaded

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Bundle not found: {BUNDLE_PATH}")

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

BUNDLE_SHA256 = sha256_file(BUNDLE_PATH)
EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="k3mini-colab-", dir="/content")).resolve()

def checked_destination(member_name):
    target = (EXTRACT_ROOT / member_name).resolve()
    if target != EXTRACT_ROOT and EXTRACT_ROOT not in target.parents:
        raise ValueError(f"Archive member escapes extraction root: {member_name!r}")
    return target

if tarfile.is_tarfile(BUNDLE_PATH):
    with tarfile.open(BUNDLE_PATH, "r:*") as tf:
        members = tf.getmembers()
        for member in members:
            checked_destination(member.name)
            if not (member.isfile() or member.isdir()):
                raise ValueError(f"Links and special archive members are not allowed: {member.name!r}")
        tf.extractall(EXTRACT_ROOT, members=members, filter="data")
elif zipfile.is_zipfile(BUNDLE_PATH):
    with zipfile.ZipFile(BUNDLE_PATH) as zf:
        for info in zf.infolist():
            checked_destination(info.filename)
            unix_mode = (info.external_attr >> 16) & 0o170000
            if unix_mode and stat.S_ISLNK(unix_mode):
                raise ValueError(f"Archive symlinks are not allowed: {info.filename!r}")
        zf.extractall(EXTRACT_ROOT)
else:
    raise ValueError("Bundle must be .tar, .tar.gz, .tgz, or .zip")

REQUIRED_FILES = [
    "train_baseline.py", "config.py", "probes.py", "training_state.py",
    "data/manifest.jsonl", "data/val_books.json", "data/shards/manifest.json",
]
candidates = []
for trainer in EXTRACT_ROOT.rglob("train_baseline.py"):
    root = trainer.parent
    if all((root / rel).is_file() for rel in REQUIRED_FILES):
        candidates.append(root)
if len(candidates) != 1:
    raise RuntimeError(f"Expected exactly one repo root in bundle; found {candidates}")
REPO_ROOT = candidates[0].resolve()
train_shards = sorted((REPO_ROOT / "data/shards").glob("gutenberg_train_*.bin"))
val_shards = sorted((REPO_ROOT / "data/shards").glob("gutenberg_val_*.bin"))
if not train_shards or not val_shards:
    raise RuntimeError("Bundle must contain Gutenberg train and validation .bin shards")
shard_manifest = json.loads((REPO_ROOT / "data/shards/manifest.json").read_text(encoding="utf-8"))
expected_shards = []
shard_root = (REPO_ROOT / "data/shards").resolve()
for split in ("train_shards", "val_shards"):
    for entry in shard_manifest[split]:
        shard = (REPO_ROOT / entry["shard_path"]).resolve()
        if shard.parent != shard_root:
            raise ValueError(f"Shard path escapes data/shards: {entry['shard_path']!r}")
        if not shard.is_file() or shard.stat().st_size != entry["file_size"]:
            raise RuntimeError(f"Shard size mismatch: {shard}")
        actual_sha256 = sha256_file(shard)
        if actual_sha256 != entry["sha256"]:
            raise RuntimeError(f"Shard SHA-256 mismatch: {shard}")
        expected_shards.append(shard)
if set(expected_shards) != set(train_shards + val_shards):
    raise RuntimeError("Archive shard files do not exactly match manifest.json")
print(f"Bundle SHA-256: {BUNDLE_SHA256}")
print(f"Repo root: {REPO_ROOT}")
print(f"Shards: {len(train_shards)} train, {len(val_shards)} validation")


In [ ]:
# 3. Install and verify the exact Linux experiment pins.
PINS = ["torch==2.13.0", "numpy==2.2.6", "tiktoken==0.11.0"]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
    "--no-cache-dir", *PINS,
])
version_check = """
import importlib.metadata as im, json, torch
versions = {name: im.version(name) for name in ('torch', 'numpy', 'tiktoken')}
expected = {'torch': '2.13.0', 'numpy': '2.2.6', 'tiktoken': '0.11.0'}
if versions != expected:
    raise SystemExit(f'Pin mismatch: {versions} != {expected}')
if not torch.cuda.is_available():
    raise SystemExit('Pinned torch cannot see CUDA')
print(json.dumps({'versions': versions, 'torch_cuda': torch.version.cuda,
                  'device': torch.cuda.get_device_name(0)}, indent=2))
"""
# This fresh child process imports the newly installed Torch; no runtime restart is needed.
subprocess.check_call([sys.executable, "-c", version_check])


In [ ]:
# 4. Syntax-check the bundled sources and run config-to-shard validation.
for name in ("train_baseline.py", "config.py", "probes.py", "training_state.py"):
    source = (REPO_ROOT / name).read_text(encoding="utf-8")
    compile(source, str(REPO_ROOT / name), "exec")

config_check = """
import json
from config import config_hash, manifest_hash, validate_against_shards
m = validate_against_shards()
print(json.dumps({'config_hash': config_hash(), 'manifest_hash': manifest_hash(),
                  'train_tokens': m['total_train_tokens'],
                  'val_tokens': m['total_val_tokens'],
                  'tokenizer': m['tokenizer'], 'vocab_size': m['vocab_size']}, indent=2))
"""
subprocess.check_call([sys.executable, "-c", config_check], cwd=REPO_ROOT)
print("Source and config validation passed.")


In [ ]:
# 5. Run the exact modal_app.py L4 smoke environment and always preserve artifacts.
import os, uuid
from datetime import datetime, timezone

SMOKE_ENV = {
    "SMOKE": "1",
    "TRAIN_STEPS": "100",
    "COMPILE": "1",
    "BATCH_SIZE": "524288",
    "MBS": "8",
    "VAL_TOKENS": "524288",
}
env = os.environ.copy()
env.update(SMOKE_ENV)
console_log = REPO_ROOT / "smoke_console.log"
proc = None
return_code = -1
caught = None

try:
    with console_log.open("w", encoding="utf-8", buffering=1) as out:
        proc = subprocess.Popen(
            [sys.executable, "-u", "train_baseline.py"],
            cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            out.write(line)
        return_code = proc.wait()
except BaseException as exc:
    caught = exc
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    if proc is not None:
        return_code = proc.returncode
finally:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    output_root = Path(DRIVE_OUTPUT_ROOT) if PERSIST_TO_DRIVE else Path("/content/k3mini-smoke-artifacts")
    ARTIFACT_DIR = output_root / stamp
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
    for src in (REPO_ROOT / "runs", REPO_ROOT / "logs"):
        if src.exists():
            shutil.copytree(src, ARTIFACT_DIR / src.name)
    if console_log.exists():
        shutil.copy2(console_log, ARTIFACT_DIR / console_log.name)
    metadata = {
        "utc": datetime.now(timezone.utc).isoformat(),
        "bundle": str(BUNDLE_PATH), "bundle_sha256": BUNDLE_SHA256,
        "repo_root": str(REPO_ROOT), "gpu": GPU_INFO,
        "smoke_env": SMOKE_ENV, "return_code": return_code,
        "exception": None if caught is None else repr(caught),
    }
    (ARTIFACT_DIR / "colab_run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"\nArtifacts preserved at: {ARTIFACT_DIR}")

if caught is not None:
    raise caught
if return_code != 0:
    raise RuntimeError(f"train_baseline.py exited with status {return_code}; artifacts were preserved")
print("100-step CUDA smoke completed successfully.")


In [ ]:
# 6. Machine-validate completion, then show concise tails from the preserved copy.
import math

index_path = ARTIFACT_DIR / "runs/index.jsonl"
if not index_path.is_file():
    raise FileNotFoundError(f"Run registry missing: {index_path}")
records = [json.loads(line) for line in index_path.read_text(encoding="utf-8").splitlines() if line.strip()]
if not records:
    raise RuntimeError("Run registry is empty")
latest = records[-1]
assert latest.get("steps_completed") == 100, latest
assert latest.get("train_steps") == 100, latest
assert latest.get("batch_size") == 524288, latest
assert latest.get("mbs") == 8, latest
assert latest.get("val_tokens") == 524288, latest
assert latest.get("world_size") == 1, latest
assert latest.get("compile") is True, latest
assert str(latest.get("torch_version")).split("+", 1)[0] == "2.13.0", latest
assert math.isfinite(float(latest.get("final_val_loss"))), latest

run_id = latest.get("run_id")
if not isinstance(run_id, str) or not run_id:
    raise RuntimeError(f"Latest registry record has no run_id: {latest}")
local_run_dir = REPO_ROOT / "runs" / run_id
saved_run_dir = ARTIFACT_DIR / "runs" / run_id
final_checkpoint = local_run_dir / "ckpt_00100.pt"
rank0_sidecar = local_run_dir / "ckpt_00100.rank00000.pt"
saved_checkpoint = saved_run_dir / final_checkpoint.name
saved_sidecar = saved_run_dir / rank0_sidecar.name
for local, saved in ((final_checkpoint, saved_checkpoint), (rank0_sidecar, saved_sidecar)):
    if not local.is_file() or local.stat().st_size == 0:
        raise FileNotFoundError(f"Required final checkpoint artifact missing: {local}")
    if not saved.is_file() or saved.stat().st_size != local.stat().st_size:
        raise RuntimeError(f"Persistent checkpoint copy is missing or incomplete: {saved}")

checkpoint_check = """
import sys, torch
common = torch.load(sys.argv[1], map_location='cpu', weights_only=True, mmap=True)
rank0 = torch.load(sys.argv[2], map_location='cpu', weights_only=True, mmap=True)
assert common.get('format') == 3, common.get('format')
assert common.get('step') == 100 and common.get('world_size') == 1
assert rank0.get('format') == 3, rank0.get('format')
assert rank0.get('step') == 100
assert rank0.get('rank') == 0 and rank0.get('world_size') == 1
"""
subprocess.check_call([sys.executable, "-c", checkpoint_check,
                       str(final_checkpoint), str(rank0_sidecar)])
print(f"PASS: run {run_id}, 100 steps, val_loss={latest['final_val_loss']}, format-3 checkpoint pair present")

def show_tail(path, chars=12000):
    text = path.read_text(encoding="utf-8", errors="replace")
    print(f"\n===== {path.relative_to(ARTIFACT_DIR)} (last {min(chars, len(text))} chars) =====")
    print(text[-chars:])

saved_console = ARTIFACT_DIR / "smoke_console.log"
if saved_console.exists():
    show_tail(saved_console)
for pattern in ("runs/*/samples.log", "logs/*.txt"):
    matches = sorted(ARTIFACT_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    if matches:
        show_tail(matches[-1], chars=8000)

checkpoints = sorted((ARTIFACT_DIR / "runs").glob("*/ckpt_*.pt")) if (ARTIFACT_DIR / "runs").exists() else []
print("\nCheckpoint files:")
for path in checkpoints:
    print(f"  {path.relative_to(ARTIFACT_DIR)}  {path.stat().st_size / 1e9:.3f} GB")
print(f"\nPersistent artifact directory: {ARTIFACT_DIR}")
